In [3]:
import dask.dataframe as dd
import pandas as pd
import numpy as np

In [4]:
dtypes = {
    'aggregate': 'object',
    'cases': 'float64',
    'city': 'object',
    'population': 'float64',
    'deaths': 'float64',
    'country': 'object',
    'state': 'object'
}

In [5]:
df = dd.read_csv('timeseries.csv', dtype = dtypes)
us_states_df = df[df['country'] == 'United States']
us_states_df['date'] = dd.to_datetime(us_states_df['date'])

# Filter for the specified date range
mask = (us_states_df['date'] >= '2020-01-01') & (us_states_df['date'] <= '2021-02-28')
date_filtered_df = us_states_df.loc[mask]

# Calculate total deaths during the period for each state
deaths_by_state = date_filtered_df.groupby('state')['deaths'].agg(['min', 'max'])
total_deaths = deaths_by_state['max'] - deaths_by_state['min']

# Calculate average population for each state during the period
avg_population = date_filtered_df.groupby('state')['population'].mean()

# Calculate per capita mortality
per_capita_mortality = total_deaths / avg_population

# Convert to pandas for final sorting and display
per_capita_mortality = per_capita_mortality.compute()

# Sort states by per capita mortality in descending order
ranked_states = per_capita_mortality.sort_values(ascending=False)

ranked_states

C:\Users\91849\anaconda3\Lib\site-packages\dask\dataframe\io\csv.py:195: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)


state
New York                        0.041343
Michigan                        0.022620
Louisiana                       0.021233
Illinois                        0.018734
Georgia                         0.018691
New Jersey                      0.018061
Indiana                         0.016409
Pennsylvania                    0.016298
Mississippi                     0.014396
Virginia                        0.013483
Ohio                            0.010005
Iowa                            0.009625
Minnesota                       0.009625
Colorado                        0.007912
Texas                           0.007733
Massachusetts                   0.007666
Missouri                        0.007651
Kentucky                        0.006805
Alabama                         0.006478
Maryland                        0.006194
North Carolina                  0.005792
Florida                         0.005433
Connecticut                     0.005287
Nebraska                        0.004420
Tennessee 

In [6]:
# Add month-year column for grouping
date_filtered_df['month_year'] = date_filtered_df['date'].dt.strftime('%Y-%m')
monthly_stats = date_filtered_df.groupby(['state', 'month_year'])\
    .agg({'cases': 'last', 'deaths': 'last'})\
    .compute()

monthly_stats['cfr'] = monthly_stats['deaths'] / monthly_stats['cases']
cfr_matrix = monthly_stats.reset_index()
cfr_pivot = cfr_matrix.pivot(index='state', columns='month_year', values='cfr')

monthly_changes = cfr_pivot.diff(axis=1)

total_changes = monthly_changes.abs().sum(axis=1)

ranked_states = total_changes.sort_values(ascending=False)

print(cfr_pivot.head())

#State Rankings by Total CFR Change
for state, change in ranked_states.items():
    print(f"{state}: {change:.6f}")

C:\Users\91849\anaconda3\Lib\site-packages\dask\dataframe\io\csv.py:195: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = reader(bio, **kwargs)



CFR Matrix (first few rows):
month_year      2020-01  2020-02   2020-03   2020-04   2020-05   2020-06  \
state                                                                      
Alabama             NaN      NaN  0.013013  0.038325  0.035094  0.024970   
Alaska              NaN      NaN  0.090909  0.375000  0.017094  0.010705   
American Samoa      NaN      NaN       NaN       NaN       NaN       NaN   
Arizona             NaN      NaN       NaN  0.041841  0.045295  0.014934   
Arkansas            NaN      NaN  0.014184  0.018740  0.018825  0.013462   

month_year       2020-07  
state                     
Alabama         0.022911  
Alaska          0.014401  
American Samoa       NaN  
Arizona         0.018442  
Arkansas        0.012323  

State Rankings by Total CFR Change:
Massachusetts: 11.088297
Rhode Island: 2.215746
Alaska: 0.652083
Utah: 0.406801
Washington: 0.269863
Michigan: 0.130247
Northern Mariana Islands: 0.078341
Connecticut: 0.077161
New York: 0.074128
Missouri: 0.070